In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
inst = {
    "ACCESS-CM2": "CSIRO-ARCCSS", "ACCESS-ESM1-5": "CSIRO", "AWI-CM-1-1-MR": "AWI", "BCC-CSM2-MR": "BCC", "CAMS-CSM1-0": "CAMS",
    "CanESM5": "CCCma", "CESM2": "NCAR", "CESM2-WACCM": "NCAR", "CMCC-CM2-SR5": "CMCC", "CNRM-CM6-1": "CNRM-CERFACS", "CNRM-CM6-1-HR": "CNRM-CERFACS",
    "CNRM-ESM2-1": "CNRM-CERFACS", "EC-Earth3": "EC-Earth-Consortium", "EC-Earth3-Veg": "EC-Earth-Consortium", "FGOALS-g3": "CAS",
    "GFDL-CM4": "NOAA-GFDL", "GFDL-ESM4": "NOAA-GFDL", "HadGEM3-GC31-LL": "MOHC", "IITM-ESM": "CCCR-IITM", "INM-CM4-8": "INM", "INM-CM5-0": "INM",
    "IPSL-CM6A-LR": "IPSL", "KACE-1-0-G": "NIMS-KMA", "KIOST-ESM": "KIOST", "MIROC6": "MIROC", "MIROC-ES2L": "MIROC", "MPI-ESM-1-2-HAM": "HAMMOZ-Consortium",
    "MPI-ESM1-2-HR": "MPI-M", "MPI-ESM1-2-LR": "MPI-M", "MRI-ESM2-0": "MRI", "NESM3": "NUIST", "NorESM2-LM": "NCC", "NorESM2-MM": "NCC",
    "TaiESM1": "AS-RCEC", "UKESM1-0-LL": "MOHC", }

grid = {
    "ACCESS-CM2": "gn", "ACCESS-ESM1-5": "gn", "AWI-CM-1-1-MR": "gn", "BCC-CSM2-MR": "gn", "CAMS-CSM1-0": "gn", "CanESM5": "gn",
    "CESM2": "gn", "CESM2-WACCM": "gn", "CMCC-CM2-SR5": "gn", "CNRM-CM6-1": "gr", "CNRM-CM6-1-HR": "gr", "CNRM-ESM2-1": "gr",
    "EC-Earth3": "gr", "EC-Earth3-Veg": "gr", "FGOALS-g3": "gn", "GFDL-CM4": "gr1", "GFDL-ESM4": "gr1", "HadGEM3-GC31-LL": "gn",
    "IITM-ESM": "gn", "INM-CM4-8": "gr1", "INM-CM5-0": "gr1", "IPSL-CM6A-LR": "gr", "KACE-1-0-G": "gr", "KIOST-ESM": "gr1",
    "MIROC-ES2L": "gn", "MIROC6": "gn", "MPI-ESM-1-2-HAM": "gn", "MPI-ESM1-2-HR": "gn", "MPI-ESM1-2-LR": "gn", "MRI-ESM2-0": "gn",
    "NESM3": "gn", "NorESM2-LM": "gn", "NorESM2-MM": "gn", "TaiESM1": "gn", "UKESM1-0-LL": "gn", }

## Generación de "inventories"

In [3]:
def generate_selection(path, df):
    with open(path, "w") as selection:
        for i, row in df.iterrows():
            if row["version"] is np.nan:
                print(f"project=CMIP6 latest=true master_id={row['instance_id'].replace('.nan', '')}\n", file=selection)
            else:
                if "," in row["version"]:
                    versions = row["version"].split(",")
                    for version in versions:
                        print(f"project=CMIP6 version={version[1:]} master_id={'.'.join(row['instance_id'].split('.')[:-1])}\n", file=selection)
                else:
                    print(f"project=CMIP6 instance_id={row['instance_id']}\n", file=selection)

### Dato diario

In [62]:
day = pd.read_csv("cmip6/CMIP6_day.csv").melt("Simulation").rename({"value": "version", "variable": "variable_id"}, axis=1)
day = (day["Simulation"]
    .str
    .split("_", expand=True)
    .rename(
        {0: "project",
         1: "source_id",
         2: "experiment_id",
         3: "member_id"},
        axis=1)
    .join(day[["variable_id", "version"]]))

day["table_id"] = day["variable_id"].map(lambda x: "day" if x != "sftlf" else "fx")
day["activity_id"] = day["experiment_id"].map(lambda x: "CMIP" if x == "historical" else "ScenarioMIP")
day["institution_id"] = day["source_id"].map(inst)
day["grid_label"] = day["source_id"].map(grid)
day["version"] = day["version"].replace('FALSE', np.nan).replace(False, np.nan)

# explode versions joined with ","
day = (day.assign(version=day["version"].str.split(","))
  .explode("version")
  .assign(version=lambda x: x["version"].str.strip())).reset_index(drop=True)

day["instance_id"] = day.apply(lambda x: '.'.join([
        "CMIP6",
        x["activity_id"],
        x["institution_id"],
        x["source_id"],
        x["experiment_id"],
        x["member_id"],
        x["table_id"],
        x["variable_id"],
        x["grid_label"],
        str(x["version"]),
    ]),
    axis=1)
day["master_id"] = day["instance_id"].str.split(".").str[:-1].str.join(".")

In [5]:
generate_selection("cmip6/selection-day.txt", day)

### Dato mensual

In [6]:
mon = pd.read_csv("cmip6/CMIP6_mon.csv").melt("dataset").rename({"value": "version", "variable": "variable_id"}, axis=1).dropna(subset=["version"])
mon = (mon["dataset"]
    .str
    .split("_", expand=True)
    .rename(
        {0: "project",
         1: "institution_id",
         2: "source_id",
         3: "experiment_id",
         4: "member_id",
         5: "table_id",
         6: "grid_label",},
        axis=1)
    .join(mon[["variable_id", "version"]]))

mon["activity_id"] = mon["experiment_id"].map(lambda x: "CMIP" if x == "historical" else "ScenarioMIP")

# explode versions joined with ","
mon = (mon.assign(version=mon["version"].str.split(","))
  .explode("version")
  .assign(version=lambda x: x["version"].str.strip()))

mon["instance_id"] = mon.apply(lambda x: '.'.join([
        "CMIP6",
        x["activity_id"],
        x["institution_id"],
        x["source_id"],
        x["experiment_id"],
        x["member_id"],
        x["table_id"],
        x["variable_id"],
        x["grid_label"],
        str(x["version"]),
    ]),
    axis=1)

In [7]:
generate_selection("cmip6/selection-mon.txt", mon)

## Análisis

In [81]:
atlas = day[~day["version"].isnull()] # persist "day", consider that "no version" is "irreproducible"
day_esgf = pd.read_json("cmip6/esgf-day-20260129.json", lines=True)

In [95]:
pangeo_gcs = pd.read_csv("pangeo-cmip6-gcs-v20251218.csv.zip")
pangeo_aws = pd.read_csv("pangeo-cmip6-aws-v20251218.csv.zip")

pangeo_gcs["instance_id"] = pangeo_gcs.apply(lambda x: '.'.join([
        "CMIP6",
        x["activity_id"],
        x["institution_id"],
        x["source_id"],
        x["experiment_id"],
        x["member_id"],
        x["table_id"],
        x["variable_id"],
        x["grid_label"],
        "v" + str(x["version"]),
    ]),
    axis=1)

pangeo_aws["instance_id"] = pangeo_aws.apply(lambda x: '.'.join([
        "CMIP6",
        x["activity_id"],
        x["institution_id"],
        x["source_id"],
        x["experiment_id"],
        x["member_id"],
        x["table_id"],
        x["variable_id"],
        x["grid_label"],
        "v" + str(x["version"]),
    ]),
    axis=1)

pangeo_gcs["master_id"] = pangeo_gcs["instance_id"].str.split(".").str[:-1].str.join(".")
pangeo_aws["master_id"] = pangeo_aws["instance_id"].str.split(".").str[:-1].str.join(".")

In [123]:
print(
    "\n".join([
        "Atlas datasets: {} ({} versioned datasets, {} without ESGF version included)".format(
            len(day), (~day["version"].isnull()).sum(), day["version"].isnull().sum()),

        "ESGF datasets: {} (matching Interactive Atlas vesions)".format(
            day["instance_id"].isin(day_esgf["instance_id"].values).sum()),

        "Pangeo GCS datasets: {} (matching Interactive Atlas vesions)".format(
            day["instance_id"].isin(pangeo_gcs["instance_id"].values).sum()),

        "Pangeo AWS datasets: {} (matching Interactive Atlas vesions)".format(
            day["instance_id"].isin(pangeo_aws["instance_id"].values).sum())]))

Atlas datasets: 1020 (787 versioned datasets, 233 without ESGF version included)
ESGF datasets: 745 (matching Interactive Atlas vesions)
Pangeo GCS datasets: 655 (matching Interactive Atlas vesions)
Pangeo AWS datasets: 673 (matching Interactive Atlas vesions)


# Diferencia master_id de instance_id

Check how many "no version" datasets are available with a "latest" version.

| Repository | Total number of datasets | Datasets with matching versions |
| --- | ----------- | --- |
| Interactive Atlas inventories | 1020 | 787 (-233) |
| ESGF | 879 | 745 (-134) |
| Pangeo (Google) | 770 | 655 (-115) |
| Pangeo (Amazon) | 797 | 673 (-124) |

In [128]:
len(day[day["version"].isnull()].drop_duplicates("master_id").set_index("master_id").drop("version", axis=1).join(
    day_esgf[["version", "master_id"]].drop_duplicates("master_id").set_index("master_id"), how="inner"))

132

In [120]:
day_esgf["master_id"] #.drop_duplicates()

0       CMIP6.CMIP.CSIRO-ARCCSS.ACCESS-CM2.historical....
1       CMIP6.CMIP.CSIRO-ARCCSS.ACCESS-CM2.historical....
2       CMIP6.ScenarioMIP.CSIRO-ARCCSS.ACCESS-CM2.ssp1...
3       CMIP6.ScenarioMIP.CSIRO-ARCCSS.ACCESS-CM2.ssp1...
4       CMIP6.ScenarioMIP.CSIRO-ARCCSS.ACCESS-CM2.ssp2...
                              ...                        
1759    CMIP6.CMIP.AS-RCEC.TaiESM1.historical.r1i1p1f1...
1760    CMIP6.CMIP.AS-RCEC.TaiESM1.historical.r1i1p1f1...
1761    CMIP6.ScenarioMIP.AS-RCEC.TaiESM1.ssp585.r1i1p...
1762    CMIP6.ScenarioMIP.AS-RCEC.TaiESM1.ssp585.r1i1p...
1763    CMIP6.ScenarioMIP.AS-RCEC.TaiESM1.ssp585.r1i1p...
Name: master_id, Length: 1764, dtype: object

In [126]:
day[day["version"].isnull()]["master_id"].drop_duplicates().isin(day_esgf[["master_id", "version"]].drop_duplicates()["master_id"]).sum()

np.int64(132)

In [112]:
day[day["version"].isnull()]["master_id"]

10      CMIP6.CMIP.AWI.AWI-CM-1-1-MR.historical.r1i1p1...
53      CMIP6.ScenarioMIP.CNRM-CERFACS.CNRM-CM6-1-HR.s...
54      CMIP6.ScenarioMIP.CNRM-CERFACS.CNRM-CM6-1-HR.s...
179     CMIP6.CMIP.AWI.AWI-CM-1-1-MR.historical.r1i1p1...
189     CMIP6.CMIP.CAMS.CAMS-CSM1-0.historical.r2i1p1f...
                              ...                        
1015    CMIP6.CMIP.MOHC.UKESM1-0-LL.historical.r1i1p1f...
1016    CMIP6.ScenarioMIP.MOHC.UKESM1-0-LL.ssp126.r1i1...
1017    CMIP6.ScenarioMIP.MOHC.UKESM1-0-LL.ssp245.r1i1...
1018    CMIP6.ScenarioMIP.MOHC.UKESM1-0-LL.ssp370.r1i1...
1019    CMIP6.ScenarioMIP.MOHC.UKESM1-0-LL.ssp585.r1i1...
Name: master_id, Length: 233, dtype: object

In [101]:
(len(day[day["version"].isnull()].set_index("master_id").drop("version", axis=1).join(
    day_esgf[["version", "master_id"]].drop_duplicates().set_index("master_id"), how="inner")) + 
day["instance_id"].isin(day_esgf["instance_id"].values).sum())

np.int64(879)

In [104]:
len(day[day["version"].isnull()].set_index("master_id").drop("version", axis=1).join(
    pangeo_gcs[["version", "master_id"]].drop_duplicates().set_index("master_id"), how="inner")) + day["instance_id"].isin(pangeo_gcs["instance_id"].values).sum()

np.int64(770)

In [98]:
len(day[day["version"].isnull()].set_index("master_id").drop("version", axis=1).join(
    pangeo_gcs[["version", "master_id"]].drop_duplicates().set_index("master_id"), how="inner"))

115

In [106]:
len(day[day["version"].isnull()].set_index("master_id").drop("version", axis=1).join(
    pangeo_aws[["version", "master_id"]].drop_duplicates().set_index("master_id"), how="inner")) +  day["instance_id"].isin(pangeo_aws["instance_id"].values).sum()

np.int64(797)